<a href="https://colab.research.google.com/github/siktamondal99-dot/Disease-Prediction-Using-Gene-Expression-Data/blob/main/Copy_of_Untitled14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load dataset
data = pd.read_csv("GSE159956_ML_dataset.csv")

# Drop useless columns BEFORE transpose
data = data.drop(columns=["Unnamed: 0", "ID_REF"])

# Transpose (genes → columns, samples → rows)
data = data.T

# Reset index (sample names)
data.reset_index(inplace=True)

print("Shape after transpose:", data.shape)

Shape after transpose: (295, 24480)


In [ ]:
# Remove sample ID column
X = data.drop(columns=['index'])

In [ ]:
data.head()

,index,0,1,2,3,4,5,6,7,8,...,24469,24470,24471,24472,24473,24474,24475,24476,24477,24478
0,GSM4851439,0.051,-0.111,-0.050,-0.341,-0.039,-0.027,-0.048,-0.014,-0.107,...,0.073,-0.369,0.068,0.070,0.040,-0.005,0.075,-0.008,0.154,-0.006
1,GSM4851440,-0.200,-0.111,-0.135,0.027,-0.166,0.130,0.179,0.028,0.417,...,-0.154,-0.643,-0.034,0.026,0.065,0.062,-0.060,-0.043,-0.014,0.132
2,GSM4851441,-0.156,-0.085,-0.179,-0.490,-0.194,-0.144,-0.074,-0.084,0.144,...,-0.067,0.258,0.243,0.198,-0.035,0.170,0.089,0.319,-0.199,0.237
3,GSM4851442,-0.206,-0.052,-0.050,-0.306,0.062,-0.012,-0.147,0.009,0.173,...,0.232,-0.251,0.199,0.040,-0.053,0.144,-0.015,0.078,0.863,0.145
4,GSM4851443,-0.515,-0.118,-0.087,-0.378,-0.093,-0.033,-0.012,0.005,0.217,...,-0.121,-0.677,0.114,0.017,-0.044,-0.015,0.101,0.002,0.134,0.024


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

# Impute missing values in X_scaled
imputer = SimpleImputer(strategy='mean') # You can choose 'median' or 'most_frequent' as well
X_scaled_imputed = imputer.fit_transform(X_scaled)

# Apply KMeans
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10) # Added n_init to suppress future warning
clusters = kmeans.fit_predict(X_scaled_imputed)

# Add cluster as target
data['target'] = clusters

print(data['target'].value_counts())

target
1    265
0     30
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_scaled_imputed, clusters)
print("Silhouette Score:", score)

Silhouette Score: 0.2590420790771542


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# X = data.drop(columns=['target', 'index']) # This X likely contains NaNs
# Instead, use the already imputed and scaled data for feature selection

selector = SelectKBest(score_func=f_classif, k=100)  # try 50/100/200
X_selected = selector.fit_transform(X_scaled_imputed, data['target'])

In [ ]:
from sklearn.model_selection import train_test_split

y = data['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential()

model.add(Dense(256, activation='relu', input_shape=(X_train.shape[1],)))
model.add(BatchNormalization())
model.add(Dropout(0.4))

model.add(Dense(128, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.7128 - loss: 0.6240 - val_accuracy: 0.6458 - val_loss: 0.6624
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8404 - loss: 0.3495 - val_accuracy: 0.7708 - val_loss: 0.5176
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9096 - loss: 0.2495 - val_accuracy: 0.8333 - val_loss: 0.3825
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9415 - loss: 0.1894 - val_accuracy: 0.9583 - val_loss: 0.2816
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9521 - loss: 0.1378 - val_accuracy: 0.9792 - val_loss: 0.2222
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9415 - loss: 0.1745 - val_accuracy: 0.9792 - val_loss: 0.1644
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9787 - loss: 0.0762 - val_accuracy: 0.9792 - val_loss: 0.1309
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9574 - loss: 0.1196 - val_accuracy: 0.9792

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=30, batch_size=8)

Epoch 1/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9534 - loss: 0.0994
Epoch 2/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9576 - loss: 0.1149
Epoch 3/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9576 - loss: 0.1025
Epoch 4/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9788 - loss: 0.0567
Epoch 5/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9746 - loss: 0.0730
Epoch 6/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9746 - loss: 0.0858
Epoch 7/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9788 - loss: 0.0576
Epoch 8/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9703 - loss: 0.0895
Epoch 9/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9534 - loss: 0.1091
Epoch 10/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9788 - loss: 0.0632
Epoch 11/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9576 - loss: 0.1187
Epoch 12/30
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9788 - lo

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("🔥 Final Accuracy:", acc)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9661 - loss: 0.0604 
🔥 Final Accuracy: 0.9661017060279846
